# Prediccion de Picos de Consumo Diario mediante Regresion

Este notebook tiene como objetivo la evaluacion del modelo de regresion para predecir el consumo diario por planta y SKU, y derivar la ocurrencia de picos de consumo diario.

## Definicion de Pico
Un pico de consumo ocurre cuando el consumo real supera el umbral establecido:
Consumo Real > $umbral$ * (Forecast Mensual / 20)

## Estructura de la Evaluacion
1. Carga de datos.
2. Construccion de variables con Polars.
3. Inferencia del modelo de regresion (LightGBM Regressor).
4. Analisis de metricas de regresion (MAE, RMSE) y clasificacion derivada (Precision, Recall, F1-Score).

| Categoria | Nombre de Variable | Descripcion |
| :--- | :--- | :--- |
| **Retardos (Lags)** | `lag_1`, `lag_2`, `lag_3`, `lag_5`, `lag_10` | Consumo real registrado 1, 2, 3, 5 y 10 dias atras respecto a la fecha actual. |
| **Ventanas Moviles** | `rolling_mean_X`, `rolling_std_X`, `rolling_max_X` | Media, desviacion estandar y maximo consumo en ventanas de X dias (donde X = 3, 5 o 10), calculados sobre el consumo desfasado 1 dia para evitar filtracion de datos. |
| **Calendario** | `day_of_month`, `days_to_end_of_month` | Dia del mes actual (1 a 31) y cantidad de dias restantes para finalizar el mes en curso. |
| **Desviaciones** | `daily_forecast` | Forecast diario teorico calculado como (Forecast Mensual / 20). |
| | `cum_actual_consumption_month_lag1` | Suma acumulada de consumo real en el mes actual hasta el dia de ayer. |
| | `cum_forecast_month_lag1` | Suma acumulada de forecast esperado en el mes actual hasta el dia de ayer. |
| | `cum_deviation_month_lag1` | Desviacion acumulada acumulada hasta el dia de ayer (Consumo acumulado - Forecast acumulado). |
| **Perfil SKU** | `sku_historical_peak_rate` | Tasa historica de picos del SKU calculado en los datos de entrenamiento (Picos / Registros totales). |
| | `sku_historical_cv` | Coeficiente de variacion historico (desviacion estandar / media de consumo) para el SKU. |


In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import polars as pl
import pandas as pd
import numpy as np

from src.features import build_features
from src.models import LGBMRegressor
from src.utils import transform_data, select_random_group
from src.utils import RegressionMetrics, ClassificationMetrics, ConsumptionPlotter

## 1. Carga de datos

In [2]:
# Carga del dato crudo
data_name = "historico_consumo.parquet"
data_path = os.path.join("data", data_name) if os.path.exists(os.path.join("data", data_name)) else os.path.join("..", "data", data_name)
df_raw = pl.read_parquet(data_path)

# Transformacion a formato analitico
df_transformed = transform_data(df_raw)

# Filtrar la serie de ejemplo para EDA y modelado
# df_preparado usa TODO el dataset transformado
df_preparado = df_transformed.with_columns(
    consumo_real=pl.col("consumo_anterior_to").fill_null(0.0),
    forecast_mensual=pl.col("forecast_mensual"),
    daily_forecast=pl.col("forecast_mensual") / 20.0,
).with_columns(
    target_is_peak=(
        pl.col("consumo_real") > (2.0 * pl.col("daily_forecast"))
    ).shift(-1).over(["planta", "sku"]).cast(pl.Int8),
    target_consumo_real=pl.col("consumo_real").shift(-1).over(["planta", "sku"]),
).filter(pl.col("target_is_peak").is_not_null())

# Guardar la serie de ejemplo separada solo para el EDA
df_eda_serie = df_preparado.filter(
    (pl.col("planta") == "SCVA") & (pl.col("sku") == "HP1_01_190_2290")
)


# Split cronologico antes de construir features para evitar leakage
fecha_corte = pl.date(2026, 2, 28)
df_train_raw = df_preparado.filter(pl.col("fecha") <= fecha_corte)
df_test_raw  = df_preparado.filter(pl.col("fecha") >  fecha_corte)

# Calcular estadisticas del SKU solo sobre train, aplicarlas al test
df_train, sku_stats = build_features(df_train_raw)
df_test,  _         = build_features(df_test_raw, sku_stats=sku_stats)

df_train.drop_nulls().head(3)


planta,sku,fecha,consumo_real,consumo_anterior_to,stock_planta,forecast_mensual,daily_forecast,target_is_peak,target_consumo_real,planta_PCEL,planta_SALC,planta_SALI,planta_SALM,planta_SBUR,planta_SCAN,planta_SCOC,planta_SCVA,planta_SPAL,planta_SQUA,planta_SVIG,cum_actual_consumption_month_lag1,cum_forecast_month_lag1,cum_deviation_month_lag1,lag_1,lag_2,lag_3,lag_5,lag_10,rolling_mean_3,rolling_std_3,rolling_max_3,rolling_mean_5,rolling_std_5,rolling_max_5,rolling_mean_10,rolling_std_10,rolling_max_10,day_of_month,days_to_end_of_month,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,sku_historical_peak_rate,sku_historical_cv
str,str,date,f64,f64,f64,f64,f64,i8,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""PCEL""","""HP2_01_105_2190""",2022-10-17,2.497,2.497,22.676,39.0,1.95,1,4.957,1,0,0,0,0,0,0,0,0,0,0,-20.087,13.65,-33.737,-0.0,-12.435,-7.774,2.0,0.0,-6.736333,6.282107,-0.0,-3.6418,6.179619,2.0,-1.2261,5.180857,5.215,17,14,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.055594,-297.080425
"""PCEL""","""HP2_01_105_2190""",2022-10-18,4.957,4.957,17.719,39.0,1.95,1,4.989,1,0,0,0,0,0,0,0,0,0,0,-17.59,15.6,-33.19,2.497,-0.0,-12.435,-0.0,2.611,-3.312667,7.998217,2.497,-3.5424,6.295957,2.497,-0.9764,5.305198,5.215,18,13,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.055594,-297.080425
"""PCEL""","""HP2_01_105_2190""",2022-10-19,4.989,4.989,12.73,39.0,1.95,0,-4.957,1,0,0,0,0,0,0,0,0,0,0,-12.633,17.55,-30.183,4.957,2.497,-0.0,-7.774,5.215,2.484667,2.478523,4.957,-2.551,7.302964,4.957,-0.7418,5.528628,5.215,19,12,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.055594,-297.080425


## 3. Ejecucion del Modelo


In [3]:
from src.models.lgbm_regressor import LGBMRegressor

# Definir la lista de variables predictoras (22 features básicas + variables de calendario)
feature_cols = [
    "lag_1", "lag_2", "lag_3", "lag_5", "lag_10",
    "rolling_mean_3", "rolling_std_3", "rolling_max_3",
    "rolling_mean_5", "rolling_std_5", "rolling_max_5",
    "rolling_mean_10", "rolling_std_10", "rolling_max_10",
    "day_of_month", "days_to_end_of_month",
    "daily_forecast",
    "cum_actual_consumption_month_lag1",
    "cum_forecast_month_lag1",
    "cum_deviation_month_lag1",
    "sku_historical_peak_rate",
    "sku_historical_cv"
] + [c for c in df_train.columns if c.startswith("day_of_week_") or c.startswith("month_")]

# Inicialización y entrenamiento del modelo
model = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)
model.fit(df_train, x_predictor=feature_cols, y="consumo_real")

# Predicción continua (Stage 1)
pred_consumo = model.predict(df_test)

# Clasificación derivada de picos (Stage 2)
umbral = 2.0 * (df_test["daily_forecast"].to_numpy())
pred_pico = (pred_consumo > umbral).astype(int)

## 4. Validacion


In [4]:
from src.utils.regression_metrics import RegressionMetrics
from src.utils.classification_metrics import ClassificationMetrics

# 1. Resumen del entrenamiento e importancia de variables del modelo
print("--- SUMMARY DEL MODELO (TRAINING & IMPORTANCE) ---")
print(model.summary())

# 2. Métricas de Regresión (Stage 1) en Test
print("\n--- METRICAS DE REGRESION (TEST SET) ---")
reg_metrics = RegressionMetrics(model_name="LGBMRegressor", n_features=len(feature_cols))
reg_metrics.compute(
    y_true=df_test["consumo_real"].to_numpy(),
    y_pred=pred_consumo,
    y_baseline=df_test["lag_1"].to_numpy()
)
print(reg_metrics.summary())

# 3. Métricas de Clasificación de Picos (Stage 2) en Test
print("\n--- METRICAS DE CLASIFICACION (TEST SET) ---")
clf_metrics = ClassificationMetrics(model_name="LGBMRegressor", beta=2.0)
clf_metrics.compute(
    y_true=df_test["target_is_peak"].to_numpy(),
    y_pred_proba=(pred_consumo / umbral),
    threshold=1.0
)
print(clf_metrics.summary())

# 4. Visualización interactiva para Planta y SKU reales del Parquet
# Ejemplo con Planta: "SCVA" y SKU: "KW2_01_135_2200"
fig = model.plot_fit(
    df_test, 
    plant="SCVA", 
    sku="HP1_01_190_2290",
    # save_path="plots/lgbm_forecast_eval.html"
)
fig.show()

--- SUMMARY DEL MODELO (TRAINING & IMPORTANCE) ---
                   Regression Metrics Summary — LGBMRegressor                   
  No. Observations : 394_545
  No. Features     : 39
--------------------------------------------------------------------------------
  Metric                                   Value
--------------------------------------------------------------------------------
  MAE                                     5.1050
  RMSE                                    7.6975
  WAPE                                    0.9251
  MAPE                                    1.8151
  Bias (mean residual)                   -0.0000
  R-squared                               0.2925
  Adj. R-squared                          0.2924

  Dep. Variable : consumo_real                    Date/Time: 2026-05-31 13:43:49
                        Feature Importances (Gain-based)                        
--------------------------------------------------------------------------------
 Feature         

C:\Users\kike\AppData\Local\Temp\ipykernel_35336\4264265345.py:23: RuntimeWarning: divide by zero encountered in divide
  y_pred_proba=(pred_consumo / umbral),


                 Classification Metrics Summary — LGBMRegressor                 
  No. Observations : 28_409
  Positives (peaks): 2_783  (9.80% prevalence)
  Decision threshold: 1.0000   |   F-beta (beta=2.0): 0.0172
--------------------------------------------------------------------------------
  Confusion Matrix              
  TP             39    FP            137
  FN           2744    TN          25489
--------------------------------------------------------------------------------
  Metric                                   Value
--------------------------------------------------------------------------------
  Precision                               0.2216
  Recall (Sensitivity)                    0.0140
  Specificity                             0.9947
  F1-Score                                0.0264
  F-beta Score (beta=2.0)              0.0172
  Accuracy                                0.8986
--------------------------------------------------------------------------------
  RO

In [5]:
df_preparado.filter(
    pl.col("planta") == "SCVA",
    pl.col("sku") == "HP1_01_190_2290"
).tail()

planta,sku,fecha,consumo_real,consumo_anterior_to,stock_planta,forecast_mensual,daily_forecast,target_is_peak,target_consumo_real,planta_PCEL,planta_SALC,planta_SALI,planta_SALM,planta_SBUR,planta_SCAN,planta_SCOC,planta_SCVA,planta_SPAL,planta_SQUA,planta_SVIG
str,str,date,f64,f64,f64,f64,f64,i8,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""SCVA""","""HP1_01_190_2290""",2026-05-20,9.076,9.076,9.086,45.0,2.25,1,9.086,0,0,0,0,0,0,0,1,0,0,0
"""SCVA""","""HP1_01_190_2290""",2026-05-21,9.086,9.086,0.0,45.0,2.25,0,-0.0,0,0,0,0,0,0,0,1,0,0,0
"""SCVA""","""HP1_01_190_2290""",2026-05-22,-0.0,-0.0,0.0,45.0,2.25,0,-0.0,0,0,0,0,0,0,0,1,0,0,0
"""SCVA""","""HP1_01_190_2290""",2026-05-25,-0.0,-0.0,0.0,45.0,2.25,0,-0.0,0,0,0,0,0,0,0,1,0,0,0
"""SCVA""","""HP1_01_190_2290""",2026-05-26,-0.0,-0.0,0.0,45.0,2.25,0,-12.698,0,0,0,0,0,0,0,1,0,0,0


In [6]:
# 1. Convertir a Pandas
df_pd = df_test.to_pandas()  # O df_preparado

# 2. Generar y asignar las predicciones del modelo
df_pd["pred_consumo"] = model.predict(df_pd)

# 3. Calcular y asignar el umbral dinámico (2x el forecast diario)
df_pd["umbral_pico"] = 2.0 * (df_pd["forecast_mensual"] / 20.0)

# 4. Ahora sí, graficar sin errores de clave
from src.utils.plotter import ConsumptionPlotter

plotter = ConsumptionPlotter(model_name="LGBMRegressor")
fig = plotter.plot_forecast(
    df=df_pd,
    date_col="fecha",
    actual_col="consumo_real",
    pred_col="pred_consumo",
    threshold_col="umbral_pico",
    plant="SCVA",
    sku="HP1_01_190_2290"
)
fig.show()
